### Установка библиотек

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split

import random, json, kagglehub, os
import numpy as np
import pandas as pd
from pathlib import Path

from transformers import BertTokenizerFast, BertForSequenceClassification, TrainingArguments, Trainer, get_cosine_schedule_with_warmup

In [2]:
def set_random_seed(seed):
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

set_random_seed(224)

### Загрузка датасета для детекции токсичности

In [3]:
path = kagglehub.dataset_download("blackmoon/russian-language-toxic-comments")
path = os.path.join(path, "labeled.csv")

### Деление на тренировочную, валидационную и тестовую выборки

In [4]:
data = pd.read_csv(path)
ds_train, ds_test = train_test_split(data, test_size=0.1)
ds_val, ds_test = train_test_split(ds_test, test_size=0.5)

### Токенизатор

In [5]:
tokenizer = BertTokenizerFast.from_pretrained('DeepPavlov/rubert-base-cased')

### Реализация срезов

In [6]:
def slice_token(index, sentences, labels, tokenizer, max_length):
    start, stop, step = index.indices(len(sentences))
    result = []
    for i in range(start, stop, step):
        encoding = tokenizer(
                [sentences[i]],
                padding='max_length',
                truncation = True,
                max_length = max_length,
                return_tensors = 'pt'
            )
        item = {key: val.squeeze(0) for key, val in encoding.items()} 
        item['labels'] = torch.tensor(labels[i], dtype=torch.long) 
        result.append(item)
    return result

### Кастомный датасет

In [ ]:
class ToxicDataset(Dataset):
    def __init__(self, sentences, labels, tokenizer, max_length):
        self.sentences = sentences.tolist()
        self.labels = labels.astype(float).tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self):
        return len(self.sentences)
    def __getitem__(self, idx):
        if isinstance(idx, slice):
            return slice_token(idx, self.sentences, self.labels, self.tokenizer, self.max_length)
        elif isinstance(idx, int):
            tokens = self.sentences[idx]
            tag = self.labels[idx]
            
            encoding = self.tokenizer(
                [tokens],
                padding='max_length',
                truncation = True,
                max_length = self.max_length,
                return_tensors = 'pt'
            )
            item = {key: val.squeeze(0) for key, val in encoding.items()} 
            item['labels'] = torch.tensor(tag, dtype=torch.long)
            return item
           

### Параметры

In [8]:
batch_size = 16
max_length = 512
epochs = 3
learning_rate = 2e-5
weight_decay = 0.01

### Создание датасетов для тренировки, валидации и тестирования

In [9]:
dataset_train = ToxicDataset(ds_train['comment'], ds_train['toxic'], tokenizer, max_length)
dataset_test = ToxicDataset(ds_test['comment'], ds_test['toxic'], tokenizer, max_length)
dataset_val = ToxicDataset(ds_val['comment'], ds_val['toxic'], tokenizer, max_length)

### Даталоадеры

In [10]:
test_loader = DataLoader(dataset_test, batch_size)
train_loader = DataLoader(dataset_train, batch_size)
val_loader = DataLoader(dataset_val, batch_size)

### Предобученная модель DeepPavlov/rubert-base-cased

In [11]:
num_labels = len(set(ds_train['toxic']))
model = BertForSequenceClassification.from_pretrained('DeepPavlov/rubert-base-cased', num_labels=num_labels)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 402.13it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.

### Функция для подсчета метрик

In [12]:
def compute_metrics(p):

    predictions, labels = p

    predictions = predictions.argmax(axis=-1)
    cm = confusion_matrix(labels, predictions)

    print("Confusion Matrix:\n", cm)

    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='macro')
    acc = accuracy_score(labels, predictions)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

### Обучение с оптимизатором и шедулером

In [13]:
optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()))
total_steps = len(dataset_train) * epochs * batch_size

scheduler = get_cosine_schedule_with_warmup(
  optimizer,
  num_warmup_steps=total_steps*0.05,
  num_training_steps=total_steps
)

In [14]:
outputs_dir = os.makedirs("../outputs", exist_ok=True)
model_dir = os.makedirs("../model_tokenizer", exist_ok=True)
model_dir = Path("../model_tokenizer")
outputs_dir = Path("../outputs")


In [16]:
training_args = TrainingArguments(
    eval_strategy="epoch",
    learning_rate = learning_rate,
    weight_decay = weight_decay,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=epochs,
    logging_strategy="epoch",
    save_strategy="no",
    metric_for_best_model="f1"  
)

In [17]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_loader.dataset,
    eval_dataset=val_loader.dataset,
    compute_metrics=compute_metrics,
    optimizers=(optimizer, scheduler)
    )

### Тренировка и валидация

In [18]:
train_metrics = trainer.train().metrics

with open(os.path.join(outputs_dir, "train_metrics.json"), "w") as f:
    json.dump(train_metrics, f, indent=2)

eval_results = trainer.evaluate()
print(f"Evaluation Results: {eval_results}")

with open(os.path.join(outputs_dir, "eval_metrics.json"), "w") as f:
    json.dump(eval_results, f, indent=2)

trainer.save_model("../model_tokenizer")
tokenizer.save_pretrained("../model_tokenizer")

 33%|███▎      | 811/2433 [18:10<32:30,  1.20s/it]

{'loss': '0.3537', 'grad_norm': '17.88', 'learning_rate': '2.602e-05', 'epoch': '1'}



 96%|█████████▌| 44/46 [00:17<00:00,  2.45it/s]
                                                  
100%|██████████| 46/46 [00:18<00:00,  2.46it/s]
                                               

Confusion Matrix:
 [[444  41]
 [ 17 219]]
{'eval_loss': '0.2047', 'eval_accuracy': '0.9196', 'eval_f1': '0.9109', 'eval_precision': '0.9027', 'eval_recall': '0.9217', 'eval_runtime': '18.48', 'eval_samples_per_second': '39.02', 'eval_steps_per_second': '2.49', 'epoch': '1'}


 67%|██████▋   | 1622/2433 [36:46<16:14,  1.20s/it] 

...

Confusion Matrix:
 [[434  51]
 [ 13 223]]
{'eval_loss': '0.2834', 'eval_accuracy': '0.9112', 'eval_f1': '0.9029', 'eval_precision': '0.8924', 'eval_recall': '0.9199', 'eval_runtime': '18.29', 'eval_samples_per_second': '39.41', 'eval_steps_per_second': '2.515', 'epoch': '3'}
{'train_runtime': '3340', 'train_samples_per_second': '11.65', 'train_steps_per_second': '0.728', 'train_loss': '0.2617', 'epoch': '3'}



100%|██████████| 46/46 [00:17<00:00,  2.58it/s]


Confusion Matrix:
 [[434  51]
 [ 13 223]]
Evaluation Results: {'eval_loss': 0.2834065556526184, 'eval_accuracy': 0.9112343966712899, 'eval_f1': 0.9029201380122864, 'eval_precision': 0.8923929195447345, 'eval_recall': 0.9198803075310152, 'eval_runtime': 18.2709, 'eval_samples_per_second': 39.462, 'eval_steps_per_second': 2.518, 'epoch': 3.0}


Writing model shards: 100%|██████████| 1/1 [00:06<00:00,  6.90s/it]


('../model_tokenizer/tokenizer_config.json',
 '../model_tokenizer/tokenizer.json')

### Тестирование модели

In [19]:
test_results = trainer.evaluate(eval_dataset=test_loader.dataset, metric_key_prefix="test")
print(f"Evaluation Results: {test_results}")

with open("../outputs/test_metrics.json", "w") as f:
    json.dump(test_results, f, indent=2)

100%|██████████| 46/46 [00:16<00:00,  2.76it/s]

Confusion Matrix:
 [[407  52]
 [ 22 240]]
Evaluation Results: {'test_loss': 0.3433878421783447, 'test_accuracy': 0.897364771151179, 'test_f1': 0.891546329723225, 'test_precision': 0.8853178784685634, 'test_recall': 0.9013703870012806, 'test_runtime': 17.1896, 'test_samples_per_second': 41.944, 'test_steps_per_second': 2.676, 'epoch': 3.0}
